# Creating Exam Questions
In this Notebook, a pipeline is established to create a set of Exam Questions based on presentation slide decks.

To reproduce the code, you need an API Key for the LLM services from ScaDS.AI.

In [ ]:
import sys
import os

# Add the root directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
from openai import OpenAI
import os
#from caching import get_zenodo_ids_from_yaml, get_zenodo_pdfs, download_pdf
import requests
import pdfplumber
from pdf2image import convert_from_bytes
import io
import base64

my_api_key = os.environ["SCADS_KEY"]
token_limit = 131000  # Token limit for the prompt

# OpenAI client
openai_client = OpenAI(base_url="https://llm.scads.ai/v1", api_key=my_api_key)

# Find model with "Qwen" in name
for model in openai_client.models.list().data:
    model_name = model.id
    #print(model_name)
    if "Qwen" in model_name:
        print(model_name)
        #break


## Before starting, all Zenodo Record, i.e. Slide Decks, are extracted from the training material database in order to choose a presentation used for generating the questions

In [ ]:
file_url = "https://raw.githubusercontent.com/NFDI4BIOIMAGE/training/main/resources/nfdi4bioimage.yml"
yaml_file = "nfdi4bioimage.yml" 
response = requests.get(file_url)

# Download the current Training Material yaml file from the Git Repository
with open(yaml_file, "wb") as file:
    file.write(response.content)
print(f"File downloaded successfully as {yaml_file}")

# Extract the Zenodo Record IDs
zenodo_ids = get_zenodo_ids_from_yaml(yaml_file)
print(f"Found {len(zenodo_ids)} Zenodo records: {zenodo_ids}")

In [ ]:
from pdf_utilities import save_images, download_zenodo_pdf

# You can either choose a specific record from the training material, or just load the desired PDF in this repository and change the pdf_path to the corresponding filename
zenodo_record_id = "12623730"  # Change to the desired Record
pdf_number = 2  # Change to the desired PDF number

# Step 1: Download PDF
pdf_path = download_zenodo_pdf(zenodo_record_id, pdf_number, "downloaded_images")

# Step 2: Save Images
save_images("downloaded_images", pdf_path)

## First, all images (that very previously downloaded) are encoded and sent to the VLM to create one question per Slide

In [ ]:
from PIL import Image

def encode_image(image_path, max_size=(512, 512), quality=75, convert_to_jpeg=True):
    """
    Resize and compress an image before encoding it to Base64.
    
    Parameters:
    - image_path (str): Path to the image.
    - max_size (tuple): Maximum width and height (default: 512x512).
    - quality (int): JPEG quality (1-100), lower = smaller size.
    - convert_to_jpeg (bool): Convert PNG to JPEG to reduce size.

    Returns:
    - str: Base64 encoded string of the optimized image.
    """
    with Image.open(image_path) as img:
        # Convert PNG to JPEG (optional)
        if convert_to_jpeg and img.format == "PNG":
            img = img.convert("RGB")  # Remove alpha channel for JPEG

        # Resize image while maintaining aspect ratio
        img.thumbnail(max_size, Image.Resampling.LANCZOS)  # Use high-quality resizing

        # Save to a BytesIO buffer
        img_buffer = io.BytesIO()
        img_format = "JPEG" if convert_to_jpeg else img.format  # Save as JPEG if converting
        img.save(img_buffer, format=img_format, quality=quality, optimize=True)

        # Convert to Base64
        img_buffer.seek(0)
        return base64.b64encode(img_buffer.getvalue()).decode("utf-8")


In [ ]:
def get_image_paths(folder):
    image_paths = sorted([
        os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".png")
    ])
    return image_paths

# Example usage
image_folder = "downloaded_images"
image_files = get_image_paths(image_folder)

In [ ]:
def ask_vlm_one_by_one(image_paths, knowledge_level="intermediate"):    
    for model in openai_client.models.list().data:
        model_name = model.id
        if "Qwen/" in model_name:
            break
        
    system_prompt = f"You are an AI assistant that analyzes slide presentations and creates a set of Exam Questions from them. Formulate the questions depending on the knowledge level, to make it easier or more detailed. The level is {knowledge_level}."
    prompt = "Take a look at the Slide and suggest a Exam Question for College Students concerning the topic of the current Slide. Output ONLY the Question, no additional information or explanations. Also output EXACTELY one Question per Image."
    
    responses = [] 

    for img_path in image_paths:
        base64_image = encode_image(img_path)  # Convert image to Base64

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ]}
        ]

        # Send request
        response = openai_client.chat.completions.create(
            model=model_name,
            messages=messages
        )

        # Store response
        responses.append(response.choices[0].message.content)

    return responses 



In [ ]:
responses = ask_vlm_one_by_one(image_files, "beginner")
for i, res in enumerate(responses):
    print(f"Response for Image {i+1}: {res}")

## Second, all questions are then again passed to the VLM, asking to summarize them in order to extract questions covering the most important points.

In [ ]:
def ask_for_summary(responses_per_slide):
    openai_client = OpenAI(base_url="https://llm.scads.ai/v1", api_key=my_api_key)

    # Find model with "llama" in name
    for model in openai_client.models.list().data:
        model_name = model.id
        if "Llama" in model_name:
            break
    
    number_of_slides = len(responses_per_slide)
    max_questions = number_of_slides / 3
    
    questions = ", ".join(responses_per_slide)
    
    prompt = f"""
    Here is a list of Exam Questions : {questions}, relating to one specific deck of presentation Slides. 
    Each Slide got converted into a question. Go through the list and extract up to {max_questions} reasonable Exam Questions for College Students to that specific topic.
    Output the Questions numerated and output nothing else than those questions (no extra explanation, etc.):
    1. Question 1
    2. Questions 2
    ...

    Try to stick as closely to the topics from the list as possible and extract the key points into new questions.
    """
    
    messages = [
            {"role": "user", "content": [
                {"type": "text", "text": prompt}
            ]}
        ]

    # Send request
    response = openai_client.chat.completions.create(model=model_name,messages=messages)

    return response.choices[0].message.content

In [ ]:
summary = ask_for_summary(responses)
print(summary)

## Third, questions can also be created with another knowledge level

In [ ]:
# Test if changing the knowledge level works
responses = ask_vlm_one_by_one(image_files, "expert")
summary = ask_for_summary(responses)

In [ ]:
print(summary)